# E04 — Triangle Inequality Average (TIA) Similarity
**D-CS Paper — Claim 4**

Kerry Mitchell's TIA formula (lkm.ufm, Ultra Fractal) is the **Holcus semantic similarity metric**.

```
tia_n = (|z^p + c| - ||z^p| - |c||) / (2|c|)
      = cos(angle between z^p and c)
```

TIA over the full orbit: `mean(tia_n for n = 1..N_iter)`

At σ=½ the formula is inherently balanced — weighting **surface** (early iterations,
syntactic) and **deep** (late iterations, semantic) relationships differently.

Mitchell wrote this to colour fractals. It is the correct semantic similarity measure
in prime-hash address space — superioir to cosine similarity, which is flat.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from engines.e04_tia_similarity import run, tia_similarity, prime_hash_address, BENCHMARK_PAIRS, OMEGA_ZS

result = run(verbose=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

results = result['results']
labels  = [f"{r['w1']}\n{r['w2']}" for r in results]
tia_vals = [r['tia'] for r in results]
relation_colors = {
    'RELATED':      '#1f77b4',
    'DISTANT':      '#d62728',
    'OPERATOR-PAIR': '#ff7f0e',
    'SAME-e0':      '#2ca02c',
    'SAME-e8':      '#9467bd',
}
colors = [relation_colors.get(r['relation'], 'gray') for r in results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: TIA scores by word pair
ax = axes[0]
y_pos = range(len(labels))
ax.barh(y_pos, tia_vals, color=colors, alpha=0.85, edgecolor='white')
ax.axvline(0, color='gray', lw=0.8)
ax.axvline(result['related_mean'], color='#1f77b4', ls='--', lw=1.5,
           label=f"related mean = {result['related_mean']:.3f}")
ax.axvline(result['distant_mean'], color='#d62728', ls='--', lw=1.5,
           label=f"distant mean = {result['distant_mean']:.3f}")
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('TIA similarity score')
ax.set_title('TIA Similarity: Word Pairs in Prime-Hash Space')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)

# Right: Prime-hash address space scatter
ax2 = axes[1]
all_words = list(set(w for r in results for w in (r['w1'], r['w2'])))
addrs = {w: prime_hash_address(w) for w in all_words}
xs = [addrs[w].real for w in all_words]
ys = [addrs[w].imag for w in all_words]
ax2.scatter(xs, ys, c='steelblue', s=40, zorder=3, alpha=0.7)
for w, a in addrs.items():
    ax2.annotate(w, (a.real, a.imag), textcoords='offset points',
                 xytext=(3, 3), fontsize=7)
ax2.axvline(OMEGA_ZS, color='red', ls='--', lw=1.2, label=f'OMEGA_ZS={OMEGA_ZS:.4f}')
ax2.set_xlabel('Re(z) = γ/(γ+1)')
ax2.set_ylabel('Im(z) = γ/(2γ+1)')
ax2.set_title('Prime-Hash Address Space')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSeparation (related − distant): {result['separation']:.4f}")
print("Positive = TIA correctly distinguishes related from distant")

In [ ]:
# Interactive: test any two words
def compare(w1, w2):
    score = tia_similarity(w1, w2)
    a1 = prime_hash_address(w1)
    a2 = prime_hash_address(w2)
    dist = abs(a1 - a2)
    print(f"TIA({w1!r}, {w2!r}) = {score:+.4f}  address_dist = {dist:.4f}")
    return score

compare('mathematics', 'physics')
compare('life', 'death')
compare('sedenion', 'octonion')
compare('riemann', 'hydrogen')

## Why TIA over Cosine

Cosine similarity measures the angle between two fixed vectors — one shot, no dynamics.

TIA iterates the orbit `z → z² + c` and accumulates the mean angular relationship
over the full trajectory. Early iterations capture syntactic proximity (surface structure).
Late iterations capture semantic proximity (deep structure).

At σ=½ the formula is balanced: surface and deep contribute equally.
This is the same balance that makes σ=½ the fixed point of the engine.

Mitchell wrote it to colour fractals beautifully. The beauty was the balance.

## Sigma

| Claim | σ |
|-------|---|
| TIA formula (established mathematics) | ∞ |
| TIA at σ=½ is inherently balanced | ∞ (follows from the formula) |
| TIA as semantic similarity metric | 3.5 (structurally motivated) |
| Mitchell's independent replication | 3.0 |